# Avaliação de Resultados — ORB-SLAM3
## Aprimoramento de SLAM Monocular com Estimação de Profundidade

Lê os arquivos da pasta `results/` do repositório e gera figuras e tabelas para o artigo.

**Para avaliar um dataset específico:** mude `DATASET` na célula 2.
**Para avaliar todos os datasets:** execute a célula 9 (tabela global).

In [1]:
# ===========================================================
# CELULA 1 - Dependencias
# ===========================================================
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'evo', 'matplotlib', 'numpy', 'pandas', '-q'])

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # backend sem interface gráfica
import matplotlib.pyplot as plt
from pathlib import Path

print('Dependencias OK!')


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


Dependencias OK!


In [ ]:
# ===========================================================
# CELULA 2 - CONFIGURACOES
# Mude apenas DATASET para avaliar outro dataset
# ===========================================================

DATASET = 'fr3_office'   # <- mude aqui: fr1_desk | fr2_xyz | fr3_office

# Pasta raiz do repositorio
REPO_DIR    = os.path.expanduser('~/orbslam3_custom')
RESULTS_DIR = os.path.join(REPO_DIR, 'results', DATASET)
GROUNDTRUTH = os.path.join(RESULTS_DIR, 'groundtruth.txt')

TOTAL_FRAMES = {'fr1_desk': 573, 'fr2_xyz': 3669, 'fr3_office': 2585}

COLORS = {
    'rgbd_baseline': '#2196F3',
    'monocular':     '#FF9800',
    'midas':         '#F44336',
    'dav2_vitl':     '#4CAF50',
    'dav2_vitb':     '#9C27B0',
    'dav2_vits':     '#00BCD4',
}
LABELS = {
    'rgbd_baseline': 'RGB-D Real',
    'monocular':     'Monocular',
    'midas':         'MiDaS',
    'dav2_vitl':     'DAV2 Large',
    'dav2_vitb':     'DAV2 Base',
    'dav2_vits':     'DAV2 Small',
}
USE_CORRECT_SCALE = {'monocular', 'midas', 'dav2_vitl', 'dav2_vitb', 'dav2_vits'}

FIGS_DIR = os.path.join(RESULTS_DIR, 'figures')
os.makedirs(FIGS_DIR, exist_ok=True)

METHODS = sorted([
    d for d in os.listdir(RESULTS_DIR)
    if os.path.isdir(os.path.join(RESULTS_DIR, d)) and d != 'figures'
])

print(f'Dataset      : {DATASET}')
print(f'Ground truth : {GROUNDTRUTH}')
print(f'Metodos      : {METHODS}')
print(f'Figuras em   : {FIGS_DIR}')

Dataset      : fr1_desk
Ground truth : /home/rafael/orbslam3_custom/results/fr1_desk/groundtruth.txt
Metodos      : ['dav2_vitl', 'midas', 'monocular', 'rgbd_baseline']
Figuras em   : /home/rafael/orbslam3_custom/results/fr1_desk/figures


In [3]:
# ===========================================================
# CELULA 3 - Calcular ATE para todos os metodos
# ===========================================================
from evo.tools import file_interface
from evo.core import sync
from evo.core.metrics import PoseRelation
import evo.main_ape as main_ape

def compute_ate(gt_path, traj_path, correct_scale=False):
    try:
        traj_ref = file_interface.read_tum_trajectory_file(gt_path)
        traj_est = file_interface.read_tum_trajectory_file(traj_path)
        traj_ref, traj_est = sync.associate_trajectories(traj_ref, traj_est)
        result = main_ape.ape(
            traj_ref, traj_est,
            est_name='est',
            pose_relation=PoseRelation.translation_part,
            align=True,
            correct_scale=correct_scale
        )
        return {
            'rmse':     result.stats['rmse'],
            'mean':     result.stats['mean'],
            'median':   result.stats['median'],
            'max':      result.stats['max'],
            'std':      result.stats['std'],
            'poses':    len(traj_est.timestamps),
            'traj_ref': traj_ref,
            'traj_est': traj_est,
            'result':   result,
            'scale':    'Sim(3)' if correct_scale else 'SE(3)',
        }
    except Exception as e:
        return {'error': str(e)}

ATE = {}
print(f'=== ATE - {DATASET} ===')
for method in METHODS:
    method_dir = os.path.join(RESULTS_DIR, method)
    traj_cam = os.path.join(method_dir, 'CameraTrajectory_run1.txt')
    traj_kf  = os.path.join(method_dir, 'KeyFrameTrajectory_run1.txt')

    if os.path.exists(traj_cam) and method != 'monocular':
        traj_path, traj_type = traj_cam, 'Camera'
    elif os.path.exists(traj_kf):
        traj_path, traj_type = traj_kf, 'KeyFrame'
    else:
        print(f'  {method:15s}: arquivo nao encontrado')
        continue

    correct_scale = method in USE_CORRECT_SCALE
    res = compute_ate(GROUNDTRUTH, traj_path, correct_scale)
    res['traj_type'] = traj_type
    res['traj_path'] = traj_path
    ATE[method] = res

    if 'error' in res:
        print(f'  {method:15s}: ERRO - {res["error"]}')
    else:
        print(f'  {LABELS.get(method,method):15s} [{res["scale"]}]: RMSE={res["rmse"]*100:.2f}cm  Poses={res["poses"]}  ({traj_type})')

=== ATE - fr1_desk ===
  DAV2 Large      [Sim(3)]: RMSE=26.84cm  Poses=612  (Camera)
  MiDaS           [Sim(3)]: RMSE=68.65cm  Poses=611  (Camera)
  Monocular       [Sim(3)]: RMSE=4.53cm  Poses=124  (KeyFrame)
  RGB-D Real      [SE(3)]: RMSE=2.23cm  Poses=572  (Camera)


In [4]:
# ===========================================================
# CELULA 4 - Tabela de resultados
# ===========================================================
total = TOTAL_FRAMES.get(DATASET, 1)
rows  = []
for method, res in ATE.items():
    if 'error' in res:
        continue
    cov = res['poses'] / total * 100
    rows.append({
        'Metodo':      LABELS.get(method, method),
        'Alinhamento': res['scale'],
        'Poses':       res['poses'],
        'Cobertura':   f'{cov:.1f}%',
        'RMSE (m)':    f"{res['rmse']:.4f}",
        'Mean (m)':    f"{res['mean']:.4f}",
        'Max (m)':     f"{res['max']:.4f}",
        'Std (m)':     f"{res['std']:.4f}",
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
csv_path = os.path.join(FIGS_DIR, f'resultados_{DATASET}.csv')
df.to_csv(csv_path, index=False)
print(f'\nSalvo: {csv_path}')

    Metodo Alinhamento  Poses Cobertura RMSE (m) Mean (m) Max (m) Std (m)
DAV2 Large      Sim(3)    612    106.8%   0.2684   0.2333  0.6242  0.1328
     MiDaS      Sim(3)    611    106.6%   0.6865   0.5972  1.3005  0.3385
 Monocular      Sim(3)    124     21.6%   0.0453   0.0332  0.1862  0.0308
RGB-D Real       SE(3)    572     99.8%   0.0223   0.0198  0.0643  0.0101

Salvo: /home/rafael/orbslam3_custom/results/fr1_desk/figures/resultados_fr1_desk.csv


In [5]:
# ===========================================================
# CELULA 5 - Grafico de barras RMSE
# ===========================================================
valid = {m: r for m, r in ATE.items() if 'rmse' in r}

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_title(f'ATE RMSE por Metodo - {DATASET}', fontsize=13, fontweight='bold')

names  = [LABELS.get(m, m) for m in valid]
rmses  = [valid[m]['rmse'] * 100 for m in valid]
colors = [COLORS.get(m, '#9E9E9E') for m in valid]
scales = [valid[m]['scale'] for m in valid]

bars = ax.bar(names, rmses, color=colors, edgecolor='white', linewidth=1.5)
for bar, val, sc in zip(bars, rmses, scales):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.2f}cm\n[{sc}]', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_ylabel('ATE RMSE (cm)')
ax.set_ylim(0, max(rmses) * 1.35)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()

path = os.path.join(FIGS_DIR, f'rmse_{DATASET}.pdf')
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Salvo: {path}')

Salvo: /home/rafael/orbslam3_custom/results/fr1_desk/figures/rmse_fr1_desk.pdf


/tmp/ipykernel_1173532/111601532.py:28: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [6]:
# ===========================================================
# CELULA 6 - Trajetorias 2D sobrepostas
# ===========================================================
fig, ax = plt.subplots(figsize=(10, 8))
ax.set_title(f'Trajetorias - {DATASET}', fontsize=13)

gt_plotted = False
for method, res in ATE.items():
    if 'traj_ref' not in res:
        continue
    if not gt_plotted:
        gt = res['traj_ref']
        ax.plot(gt.positions_xyz[:, 0], gt.positions_xyz[:, 1],
                'k--', lw=1.5, label='Ground Truth', alpha=0.6, zorder=1)
        gt_plotted = True
    est   = res['traj_est']
    color = COLORS.get(method, '#9E9E9E')
    label = f"{LABELS.get(method, method)} (RMSE={res['rmse']*100:.2f}cm)"
    ax.plot(est.positions_xyz[:, 0], est.positions_xyz[:, 1],
            color=color, lw=2, label=label, zorder=2)

ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()

path = os.path.join(FIGS_DIR, f'trajetorias_{DATASET}.pdf')
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Salvo: {path}')

Salvo: /home/rafael/orbslam3_custom/results/fr1_desk/figures/trajetorias_fr1_desk.pdf


/tmp/ipykernel_1173532/625413162.py:33: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [7]:
# ===========================================================
# CELULA 7 - ATE ao longo do tempo
# ===========================================================
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title(f'ATE ao Longo do Tempo - {DATASET}', fontsize=13)

for method, res in ATE.items():
    if 'result' not in res:
        continue
    errors = res['result'].np_arrays['error_array']
    ts     = res['traj_est'].timestamps
    ts     = ts - ts[0]
    color  = COLORS.get(method, '#9E9E9E')
    label  = f"{LABELS.get(method, method)} (RMSE={res['rmse']*100:.2f}cm)"
    ax.plot(ts, errors * 100, color=color, lw=1.5, alpha=0.85, label=label)

ax.set_xlabel('Tempo (s)')
ax.set_ylabel('ATE (cm)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()

path = os.path.join(FIGS_DIR, f'ate_temporal_{DATASET}.pdf')
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Salvo: {path}')

Salvo: /home/rafael/orbslam3_custom/results/fr1_desk/figures/ate_temporal_fr1_desk.pdf


/tmp/ipykernel_1173532/1581650951.py:27: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [8]:
# ===========================================================
# CELULA 8 - Cobertura da trajetoria
# ===========================================================
total = TOTAL_FRAMES.get(DATASET, 1)
valid = {m: r for m, r in ATE.items() if 'poses' in r}

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_title(f'Cobertura da Trajetoria - {DATASET} ({total} frames)', fontsize=13)

names    = [LABELS.get(m, m) for m in valid]
coverage = [valid[m]['poses'] / total * 100 for m in valid]
colors   = [COLORS.get(m, '#9E9E9E') for m in valid]

bars = ax.bar(names, coverage, color=colors, edgecolor='white', linewidth=1.5)
for bar, val, method in zip(bars, coverage, valid):
    poses = valid[method]['poses']
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%\n({poses})', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.axhline(y=100, color='gray', linestyle='--', alpha=0.4)
ax.set_ylabel('Cobertura (%)')
ax.set_ylim(0, 120)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()

path = os.path.join(FIGS_DIR, f'cobertura_{DATASET}.pdf')
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Salvo: {path}')

Salvo: /home/rafael/orbslam3_custom/results/fr1_desk/figures/cobertura_fr1_desk.pdf


/tmp/ipykernel_1173532/486760325.py:30: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [9]:
# ===========================================================
# CELULA 9 - TABELA GLOBAL (todos os datasets)
# ===========================================================

from evo.tools import file_interface
from evo.core import sync

import evo.main_ape as main_ape

REPO_DIR     = os.path.expanduser('~/orbslam3_custom')
ALL_DATASETS = ['fr1_desk', 'fr2_xyz', 'fr3_office']
TOTAL_FRAMES = {'fr1_desk': 573, 'fr2_xyz': 3669, 'fr3_office': 2585}
USE_CORRECT_SCALE = {'monocular'}

all_rows = []

for dataset in ALL_DATASETS:
    results_dir = os.path.join(REPO_DIR, 'results', dataset)
    gt_path     = os.path.join(results_dir, 'groundtruth.txt')
    total       = TOTAL_FRAMES.get(dataset, 1)

    if not os.path.exists(gt_path):
        print(f'Ground truth nao encontrado: {gt_path}')
        continue

    methods = sorted([
        d for d in os.listdir(results_dir)
        if os.path.isdir(os.path.join(results_dir, d)) and d != 'figures'
    ])

    for method in methods:
        method_dir = os.path.join(results_dir, method)
        traj_cam   = os.path.join(method_dir, 'CameraTrajectory_run1.txt')
        traj_kf    = os.path.join(method_dir, 'KeyFrameTrajectory_run1.txt')

        if os.path.exists(traj_cam) and method != 'monocular':
            traj_path, traj_type = traj_cam, 'Camera'
        elif os.path.exists(traj_kf):
            traj_path, traj_type = traj_kf, 'KeyFrame'
        else:
            continue

        correct_scale = method in USE_CORRECT_SCALE
        try:
            traj_ref = file_interface.read_tum_trajectory_file(gt_path)
            traj_est = file_interface.read_tum_trajectory_file(traj_path)
            traj_ref, traj_est = sync.associate_trajectories(traj_ref, traj_est)
            result = main_ape.ape(
                traj_ref, traj_est,
                est_name='est',
                pose_relation=PoseRelation.translation_part,
                align=True,
                correct_scale=correct_scale
            )
            poses = len(traj_est.timestamps)
            cov   = poses / total * 100
            scale = 'Sim(3)' if correct_scale else 'SE(3)'
            all_rows.append({
                'Dataset':     dataset,
                'Metodo':      LABELS.get(method, method),
                'Alinhamento': scale,
                'Poses':       poses,
                'Cobertura':   f'{cov:.1f}%',
                'RMSE (m)':    f"{result.stats['rmse']:.4f}",
                'Mean (m)':    f"{result.stats['mean']:.4f}",
                'Max (m)':     f"{result.stats['max']:.4f}",
            })
        except Exception as e:
            all_rows.append({
                'Dataset': dataset, 'Metodo': LABELS.get(method, method),
                'Alinhamento': '-', 'Poses': 0, 'Cobertura': '-',
                'RMSE (m)': f'ERRO: {e}', 'Mean (m)': '-', 'Max (m)': '-'
            })

df_all = pd.DataFrame(all_rows)
print('=== TABELA GLOBAL ===')
print(df_all.to_string(index=False))

global_figs = os.path.join(REPO_DIR, 'results', 'figures')
os.makedirs(global_figs, exist_ok=True)
csv_path = os.path.join(global_figs, 'tabela_global.csv')
df_all.to_csv(csv_path, index=False)
print(f'\nSalvo: {csv_path}')

=== TABELA GLOBAL ===
   Dataset     Metodo Alinhamento  Poses Cobertura RMSE (m) Mean (m) Max (m)
  fr1_desk DAV2 Large       SE(3)    612    106.8%   0.3514   0.3387  0.6025
  fr1_desk      MiDaS       SE(3)    611    106.6%   0.9913   0.9384  1.5237
  fr1_desk  Monocular      Sim(3)    124     21.6%   0.0453   0.0332  0.1862
  fr1_desk RGB-D Real       SE(3)    572     99.8%   0.0223   0.0198  0.0643
   fr2_xyz DAV2 Large       SE(3)   3669    100.0%   0.2118   0.1723  0.6062
   fr2_xyz      MiDaS       SE(3)   3669    100.0%   0.1863   0.1471  0.5659
   fr2_xyz  Monocular      Sim(3)     59      1.6%   0.0031   0.0028  0.0060
   fr2_xyz RGB-D Real       SE(3)   3615     98.5%   0.0040   0.0036  0.0102
fr3_office DAV2 Large       SE(3)   2583     99.9%   1.0255   0.9942  1.5360
fr3_office      MiDaS       SE(3)   2582     99.9%   1.1872   1.0996  1.8888
fr3_office  Monocular      Sim(3)    314     12.1%   0.0124   0.0106  0.0357
fr3_office RGB-D Real       SE(3)   2486     96.2%   0

In [10]:
# ===========================================================
# CELULA 10 - Grafico global comparativo (todos os datasets)
# ===========================================================
datasets = df_all['Dataset'].unique()
fig, axes = plt.subplots(1, len(datasets), figsize=(16, 5), sharey=False)
fig.suptitle('ATE RMSE por Metodo e Dataset', fontsize=14, fontweight='bold')

for ax, dataset in zip(axes, datasets):
    subset = df_all[df_all['Dataset'] == dataset].copy()
    subset = subset[~subset['RMSE (m)'].str.startswith('ERRO')]
    
    names  = subset['Metodo'].tolist()
    rmses  = [float(v) * 100 for v in subset['RMSE (m)'].tolist()]
    method_keys = [k for k, v in LABELS.items() if v in names]
    colors = [COLORS.get(k, '#9E9E9E') for k in method_keys]
    scales = subset['Alinhamento'].tolist()

    bars = ax.bar(names, rmses, color=colors[:len(names)], edgecolor='white', linewidth=1.5)
    for bar, val, sc in zip(bars, rmses, scales):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}cm', ha='center', va='bottom', fontsize=7, fontweight='bold')

    ax.set_title(dataset, fontsize=11)
    ax.set_ylabel('RMSE (cm)')
    ax.set_ylim(0, max(rmses) * 1.35 if rmses else 1)
    ax.tick_params(axis='x', rotation=20)
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
path = os.path.join(global_figs, 'rmse_global.pdf')
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Salvo: {path}')

Salvo: /home/rafael/orbslam3_custom/results/figures/rmse_global.pdf


/tmp/ipykernel_1173532/3856875423.py:34: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [11]:
# ===========================================================
# CELULA 11 - Resumo final
# ===========================================================
print('=' * 75)
print('RESUMO GLOBAL - ORB-SLAM3 com Depth Sintetico')
print('=' * 75)
print(f'{"Dataset":<12} {"Metodo":<15} {"Align":<7} {"Poses":<8} {"RMSE(cm)":<10} {"Cob%"}')
print('-' * 75)
for _, row in df_all.iterrows():
    if row['RMSE (m)'].startswith('ERRO'):
        continue
    rmse_cm = float(row['RMSE (m)']) * 100
    print(f"{row['Dataset']:<12} {row['Metodo']:<15} {row['Alinhamento']:<7} "
          f"{str(row['Poses']):<8} {rmse_cm:<10.2f} {row['Cobertura']}")
print('=' * 75)
print(f'\nFiguras salvas em: {global_figs}')
for f in sorted(os.listdir(global_figs)):
    print(f'  {f}')

RESUMO GLOBAL - ORB-SLAM3 com Depth Sintetico
Dataset      Metodo          Align   Poses    RMSE(cm)   Cob%
---------------------------------------------------------------------------
fr1_desk     DAV2 Large      SE(3)   612      35.14      106.8%
fr1_desk     MiDaS           SE(3)   611      99.13      106.6%
fr1_desk     Monocular       Sim(3)  124      4.53       21.6%
fr1_desk     RGB-D Real      SE(3)   572      2.23       99.8%
fr2_xyz      DAV2 Large      SE(3)   3669     21.18      100.0%
fr2_xyz      MiDaS           SE(3)   3669     18.63      100.0%
fr2_xyz      Monocular       Sim(3)  59       0.31       1.6%
fr2_xyz      RGB-D Real      SE(3)   3615     0.40       98.5%
fr3_office   DAV2 Large      SE(3)   2583     102.55     99.9%
fr3_office   MiDaS           SE(3)   2582     118.72     99.9%
fr3_office   Monocular       Sim(3)  314      1.24       12.1%
fr3_office   RGB-D Real      SE(3)   2486     1.26       96.2%

Figuras salvas em: /home/rafael/orbslam3_custom/results/